# MODEL DEVELOPMENT


In [52]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, f1_score, roc_auc_score, 
                            precision_score, recall_score, accuracy_score,
                            confusion_matrix, roc_curve, precision_recall_curve)
import xgboost as xgb
from scipy.stats import ttest_ind
import json
import os
import warnings
import time

warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print(" All libraries imported successfully")

# Load the dataset
df_model= pd.read_csv('../../data/processed/modeling_dataset_25k.csv')
print(f" Dataset loaded successfully with shape: {df_model.shape}")


 All libraries imported successfully
 Dataset loaded successfully with shape: (25000, 33)



### Feature Engineering Pipeline 

Objective: Transform raw data into model-ready features


In [53]:

print(f"\nStarting Dataset:")
print(f"Samples: {len(df_model):,}")
print(f"Original Features: {df_model.shape[1]}")


# STEP 1: Log Transformations (to fix skewness)
print("STEP 1: LOG TRANSFORMATIONS")

skewed_features = ['views', 'likes', 'comments', 'shares', 'saves']
for col in skewed_features:
    if col in df_model.columns:
        df_model[f'{col}_log'] = np.log1p(df_model[col])
        print(f" Created {col}_log")


# STEP 2: Engagement Rate Features (quality not quantity)
print("STEP 2: ENGAGEMENT RATES")

df_model['like_rate'] = df_model['likes'] / (df_model['views'] + 1)
df_model['comment_rate'] = df_model['comments'] / (df_model['views'] + 1)
df_model['share_rate'] = df_model['shares'] / (df_model['views'] + 1)
print(" Created like_rate, comment_rate, share_rate")


# STEP 3: Temporal Features (optimal posting times)
print("STEP 3: TEMPORAL FEATURES")

# Convert day names to numbers if needed
if df_model['posting_day'].dtype == 'object':
    day_mapping = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 
                   'Friday': 4, 'Saturday': 5, 'Sunday': 6}
    df_model['posting_day'] = df_model['posting_day'].map(day_mapping)
    
    # Check for any unmapped values
    if df_model['posting_day'].isnull().any():
        print(f"Warning: Found {df_model['posting_day'].isnull().sum()} unmapped day values")
        # Fill with most common day (weekday = 2 for Wednesday)
        df_model['posting_day'].fillna(2, inplace=True)

df_model['is_weekend'] = df_model['posting_day'].isin([5, 6]).astype(int)
df_model['is_peak_hour'] = df_model['posting_hour'].isin([18, 19, 20]).astype(int)
df_model['is_evening'] = (df_model['posting_hour'] >= 18).astype(int)
print(" Created is_weekend, is_peak_hour, is_evening")


# STEP 4: Content Optimisation Features (EDA findings)

print("STEP 4: CONTENT OPTIMISATION")


df_model['optimal_hashtag_range'] = df_model['hashtag_count'].between(5, 10).astype(int)
df_model['has_short_caption'] = (df_model['caption_length'] < 50).astype(int)
df_model['has_optimal_caption'] = df_model['caption_length'].between(100, 150).astype(int)
df_model['has_long_caption'] = (df_model['caption_length'] > 200).astype(int)
print("Created optimal_hashtag_range, caption length features")


# STEP 5: Encode Categorical Variables
print("STEP 5: CATEGORICAL ENCODING")


# Platform (One-Hot -- Binary columns for each platform)
platform_encoded = pd.get_dummies(df_model['platform'], prefix='platform', dtype=int)
df_model = pd.concat([df_model, platform_encoded], axis=1)
print(f" Platform encoded: {list(platform_encoded.columns)}")

#  Trend Label (One-Hot)
trend_encoded = pd.get_dummies(df_model['trend_label'], prefix='trend', dtype=int)
df_model = pd.concat([df_model, trend_encoded], axis=1)
print(f" Trend encoded: {list(trend_encoded.columns)}")

# Category (Target Encoding)
category_encoding = df_model.groupby('category')['engagement_binary'].mean()
df_model['category_encoded'] = df_model['category'].map(category_encoding)
print(f" Category target encoded")


# STEP 6: Final Feature Selection
print("STEP 6: FINAL FEATURE LIST")


selected_features = [
    # Trend features (CORE HYPOTHESIS)
    'has_trend',
    'trend_rising', 'trend_seasonal', 'trend_stable', 'trend_declining',
    
    # Temporal features
    'posting_hour', 'posting_day', 'posting_month',
    'is_peak_hour', 'is_weekend', 'is_evening',
    
    # Content features
    'caption_length', 'hashtag_count', 'duration_sec',
    'optimal_hashtag_range', 'has_optimal_caption', 
    'has_short_caption', 'has_long_caption',
    
    # Engagement (log-transformed)
    'views_log', 'likes_log', 'comments_log', 'shares_log', 'saves_log',
    
    # Engagement rates
    'like_rate', 'comment_rate', 'share_rate',
    
    # Platform (one-hot)
    'platform_tiktok', 'platform_instagram', 'platform_youtube',
    
    # Category (target encoded)
    'category_encoded'
]

print(f"Selected {len(selected_features)} features")

# Verify all features exist
missing = [f for f in selected_features if f not in df_model.columns]
if missing:
    print(f"  Missing: {missing}")
else:
    print(f"All features present!")


# STEP 7: Data Quality Check
print("STEP 7: QUALITY CHECKS")


# Missing values
missing_count = df_model[selected_features].isnull().sum().sum()
print(f"Missing values: {missing_count}")

# If there are missing values, show which features and fill them
if missing_count > 0:
    missing_by_feature = df_model[selected_features].isnull().sum()
    missing_by_feature = missing_by_feature[missing_by_feature > 0]
    print("\nFeatures with missing values:")
    for feat, count in missing_by_feature.items():
        print(f"  {feat}: {count}")
    print("\nFilling missing values with 0...")
    df_model[selected_features] = df_model[selected_features].fillna(0)
    print("Missing values filled!")

# Infinite values
inf_count = np.isinf(df_model[selected_features].select_dtypes(include=[np.number])).sum().sum()
print(f"Infinite values: {inf_count}")

if missing_count == 0 and inf_count == 0:
    print("Data quality: PASSED")
elif missing_count > 0:
    print("Data quality: FIXED (NaN values filled)")


# STEP 8: Save Configuration
print("STEP 8: SAVE CONFIGURATION")
os.makedirs('../../models/', exist_ok=True)

feature_config = {
    'selected_features': selected_features,
    'feature_count': len(selected_features),
    'date': '2026-03-03',
    'dataset_size': len(df_model),
    'target': 'engagement_binary'
}
with open('../../models/feature_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)

print("Configuration saved to models/feature_config.json")


print(f"\nFeature Engineering Summary:")
print(f"   Original features:    {df_model.shape[1] - len(selected_features)}")
print(f"   Engineered features:  {len(selected_features)}")
print(f"   Dataset size:         {len(df_model):,} samples")
print(f"   Target variable:      engagement_binary")
print(f"   High engagement:      {df_model['engagement_binary'].sum():,} ({df_model['engagement_binary'].mean()*100:.1f}%)")



Starting Dataset:
Samples: 25,000
Original Features: 33
STEP 1: LOG TRANSFORMATIONS
 Created views_log
 Created likes_log
 Created comments_log
 Created shares_log
 Created saves_log
STEP 2: ENGAGEMENT RATES
 Created like_rate, comment_rate, share_rate
STEP 3: TEMPORAL FEATURES
 Created is_weekend, is_peak_hour, is_evening
STEP 4: CONTENT OPTIMISATION
Created optimal_hashtag_range, caption length features
STEP 5: CATEGORICAL ENCODING
 Platform encoded: ['platform_instagram', 'platform_tiktok', 'platform_youtube']
 Trend encoded: ['trend_declining', 'trend_rising', 'trend_seasonal', 'trend_stable']
 Category target encoded
STEP 6: FINAL FEATURE LIST
Selected 30 features
All features present!
STEP 7: QUALITY CHECKS
Missing values: 9093

Features with missing values:
  duration_sec: 9093

Filling missing values with 0...
Missing values filled!
Infinite values: 0
Data quality: FIXED (NaN values filled)
STEP 8: SAVE CONFIGURATION
Configuration saved to models/feature_config.json

Feature E

### Train-Test-Validation Split (70-15-15)

In [54]:

print("TRAIN-TEST-VALIDATION SPLIT")

# Prepare features and target
X = df_model[selected_features]
y = df_model['engagement_binary']

print(f"\nTotal dataset: {len(X):,} samples")
print(f"Features: {len(selected_features)}")
print(f"Target: engagement_binary")

# First split: 70% train, 30% temp (will become val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, 
    test_size=0.30, 
    stratify=y, 
    random_state=42
)

# Second split: Split temp into 50/50 (each becomes 15% of total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.50, 
    stratify=y_temp, 
    random_state=42
)


# Verify Class Distributions
print("CLASS DISTRIBUTION (High Engagement %)")

train_high_pct = y_train.mean() * 100
val_high_pct = y_val.mean() * 100
test_high_pct = y_test.mean() * 100

print(f"Train: {len(y_train):>6,} samples | High: {train_high_pct:.1f}% | Low: {100-train_high_pct:.1f}%")
print(f"Val:   {len(y_val):>6,} samples | High: {val_high_pct:.1f}% | Low: {100-val_high_pct:.1f}%")
print(f"Test:  {len(y_test):>6,} samples | High: {test_high_pct:.1f}% | Low: {100-test_high_pct:.1f}%")


# Success Check
print("VALIDATION CHECKS")

# Check 1: Sizes are correct
expected_train = int(0.70 * len(X))
expected_val = int(0.15 * len(X))
expected_test = int(0.15 * len(X))

size_check = (
    abs(len(X_train) - expected_train) <= 10 and
    abs(len(X_val) - expected_val) <= 10 and
    abs(len(X_test) - expected_test) <= 10
)

print(f"Split sizes correct (70/15/15): {'PASS' if size_check else 'FAIL'}")

# Check 2: Class balance maintained (within 2% of original)
original_high_pct = y.mean() * 100
balance_check = (
    abs(train_high_pct - original_high_pct) < 2 and
    abs(val_high_pct - original_high_pct) < 2 and
    abs(test_high_pct - original_high_pct) < 2
)

print(f"Stratification maintained: {'PASS' if balance_check else 'FAIL'}")

# Check 3: No data overlap
overlap_check = (
    len(set(X_train.index) & set(X_val.index)) == 0 and
    len(set(X_train.index) & set(X_test.index)) == 0 and
    len(set(X_val.index) & set(X_test.index)) == 0
)

print(f"No data leakage (no overlap): {' PASS' if overlap_check else 'FAIL'}")

if size_check and balance_check and overlap_check:

    print("TASK COMPLETE!")

else:
    print("\nome checks failed - review splits before proceeding")

TRAIN-TEST-VALIDATION SPLIT

Total dataset: 25,000 samples
Features: 30
Target: engagement_binary
CLASS DISTRIBUTION (High Engagement %)
Train: 17,500 samples | High: 78.1% | Low: 21.9%
Val:    3,750 samples | High: 78.1% | Low: 21.9%
Test:   3,750 samples | High: 78.1% | Low: 21.9%
VALIDATION CHECKS
Split sizes correct (70/15/15): PASS
Stratification maintained: PASS
No data leakage (no overlap):  PASS
TASK COMPLETE!


### Baseline Model (Logistic Regression)
Objective: Establish baseline performance with simple model.

In [55]:

print("BASELINE MODEL - LOGISTIC REGRESSION")


# Train baseline model
print("\nTraining baseline model...")
start_time = time.time()

baseline = LogisticRegression(max_iter=1000, random_state=42)
baseline.fit(X_train, y_train)

training_time = time.time() - start_time
print(f"Training completed in {training_time:.2f} seconds")


# Make Predictions
print("\nMaking predictions on validation set...")
y_val_pred = baseline.predict(X_val)
y_val_proba = baseline.predict_proba(X_val)[:, 1]


# Performance Metrics
print("VALIDATION PERFORMANCE METRICS")


f1 = f1_score(y_val, y_val_pred)
roc_auc = roc_auc_score(y_val, y_val_proba)
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
accuracy = accuracy_score(y_val, y_val_pred)

print(f"\nF1 Score:       {f1:.4f}")
print(f"ROC-AUC:        {roc_auc:.4f}")
print(f"Precision:      {precision:.4f}")
print(f"Recall:         {recall:.4f}")
print(f"Accuracy:       {accuracy:.4f}")

print("\n" + "-"*70)
print("CLASSIFICATION REPORT")
print("-"*70)
print(classification_report(y_val, y_val_pred, 
                           target_names=['Low Engagement', 'High Engagement']))



# Feature Importance Analysis
print("\n" + "="*70)
print("FEATURE IMPORTANCE (Top 10 by Coefficient Magnitude)")
print("="*70)

coef_df = pd.DataFrame({
    'feature': selected_features,
    'coefficient': baseline.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print("\nTop 10 Most Important Features:")
print(coef_df.head(10).to_string(index=False))

# Check if has_trend is in top 10 (for hypothesis validation)
has_trend_rank = coef_df.reset_index(drop=True).index[coef_df['feature'] == 'has_trend'].tolist()
if has_trend_rank:
    rank = has_trend_rank[0] + 1
    print(f"\n'has_trend' ranked #{rank} out of {len(selected_features)} features")


# Success Criteria Check
criteria = {
    f"F1 Score ≥ 0.90": f1 >= 0.90,
    f"ROC-AUC ≥ 0.85": roc_auc >= 0.85,
    f"Precision ≥ 0.85": precision >= 0.85,
    f"Recall ≥ 0.90": recall >= 0.90,
    f"Training time < 60s": training_time < 60
}

for criterion, passed in criteria.items():
    status = " PASS" if passed else " FAIL"
    print(f"{criterion:<30} {status}")

all_passed = all(criteria.values())


# Final Summary
if all_passed:
    print("\n" + "="*70)
    print(" TASK 1c COMPLETE!")
    print("="*70)
    print(f"\nBaseline Performance Summary:")
    print(f"   • F1 Score:      {f1:.4f}")
    print(f"   • ROC-AUC:       {roc_auc:.4f}")
    print(f"   • Precision:     {precision:.4f}")
    print(f"   • Recall:        {recall:.4f}")
    print(f"   • Training time: {training_time:.2f}s")
else:
    print("\n  Some criteria not met - review performance before proceeding")

BASELINE MODEL - LOGISTIC REGRESSION

Training baseline model...


Training completed in 1.65 seconds

Making predictions on validation set...
VALIDATION PERFORMANCE METRICS

F1 Score:       0.9605
ROC-AUC:        0.9781
Precision:      0.9548
Recall:         0.9662
Accuracy:       0.9379

----------------------------------------------------------------------
CLASSIFICATION REPORT
----------------------------------------------------------------------
                 precision    recall  f1-score   support

 Low Engagement       0.87      0.84      0.86       821
High Engagement       0.95      0.97      0.96      2929

       accuracy                           0.94      3750
      macro avg       0.91      0.90      0.91      3750
   weighted avg       0.94      0.94      0.94      3750


FEATURE IMPORTANCE (Top 10 by Coefficient Magnitude)

Top 10 Most Important Features:
           feature  coefficient
         like_rate     6.592588
platform_instagram     4.744863
   trend_declining     4.187281
      trend_stable     4.057177
  platform_youtube  